04_plot_centroid_heatmap_and_validity.py

Tugas Orang 5 — Heatmap karakteristik centroid + grafik indeks validitas.

Input (data real dari Orang 1):
- data/processed/cluster_centroids_standardized.csv
    kolom: cluster, maternal_age_risk_z, low_knowledge_z,
           water_no_or_unimproved_z, water_limited_z,
           sanitation_babs_z, sanitation_unimproved_z

- data/processed/fcm_experiment_results.csv
    kolom: c, m, seed, partition_coefficient, xie_beni,
           modified_partition_coefficient, partition_entropy

Output:
- outputs/figures/heatmap_centroid.png
- outputs/figures/grafik_indeks_validitas.png

In [6]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

PROJECT_ROOT    = Path().resolve().parent
CENTROIDS_PATH  = PROJECT_ROOT / "outputs" / "model" / "cluster_centroids_standardized.csv"
VALIDITY_PATH   = PROJECT_ROOT / "outputs" / "model" / "fcm_experiment_results.csv"
FIG_DIR         = PROJECT_ROOT / "outputs" / "figures"

VARIABLE_LABELS: dict[str, str] = {
    "maternal_age_risk_z":      "Risiko Usia\nIbu Hamil",
    "low_knowledge_z":          "Rendahnya\nPengetahuan Stunting",
    "water_no_or_unimproved_z": "Air Minum\nTidak Layak / Tanpa Akses",
    "water_limited_z":          "Air Minum\nAkses Terbatas",
    "sanitation_babs_z":        "Sanitasi\nBABS",
    "sanitation_unimproved_z":  "Sanitasi\nTidak Layak",
}

CLUSTER_LABELS = {
    "cluster_1": "Klaster 1\n(Risiko Relatif Rendah)",
    "cluster_2": "Klaster 2\n(Risiko Relatif Tinggi)",
}

### Heatmap centroid

In [7]:
def plot_centroid_heatmap() -> None:
    df = pd.read_csv(CENTROIDS_PATH)
    df = df.set_index("cluster")

    cols = [c for c in VARIABLE_LABELS if c in df.columns]
    df   = df[cols].rename(columns=VARIABLE_LABELS)
    df.index = [CLUSTER_LABELS.get(i, i) for i in df.index]

    fig, ax = plt.subplots(figsize=(12, 4.5))
    vmax = max(abs(df.values.min()), abs(df.values.max()))

    im = ax.imshow(df.values, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")

    ax.set_xticks(range(len(df.columns)))
    ax.set_xticklabels(df.columns, fontsize=9.5)
    ax.set_yticks(range(len(df.index)))
    ax.set_yticklabels(df.index, fontsize=10)

    # Annotasi nilai z di setiap sel
    for i in range(df.shape[0]):
        for j in range(df.shape[1]):
            val = df.values[i, j]
            tcolor = "white" if abs(val) > 0.55 * vmax else "black"
            ax.text(j, i, f"{val:+.3f}", ha="center", va="center",
                    fontsize=10, color=tcolor, weight="bold")

    cbar = fig.colorbar(im, ax=ax, shrink=0.9, pad=0.02)
    cbar.set_label(
        "Z-score  (merah = lebih tinggi dari rata-rata nasional · biru = lebih rendah)",
        fontsize=9,
    )

    ax.set_title(
        "Karakteristik Centroid Tiap Klaster FCM\n"
        "(nilai z-score setelah standardisasi; merah = faktor risiko lebih tinggi)",
        fontsize=13, weight="bold", pad=12,
    )
    fig.tight_layout(pad=1.5)

    out = FIG_DIR / "heatmap_centroid.png"
    fig.savefig(out, dpi=220, bbox_inches="tight", facecolor="white")
    print(f"Heatmap centroid disimpan → {out}")
    plt.close(fig)


# ── Grafik indeks validitas ──────────────────────────────────────────────────

In [8]:
def plot_validity_indices() -> None:
    df = pd.read_csv(VALIDITY_PATH)

    # Ringkasan per (c, m): rata-rata dari semua seed
    summary = (
        df.groupby(["c", "m"])[
            ["partition_coefficient", "xie_beni",
             "modified_partition_coefficient", "partition_entropy"]
        ]
        .mean()
        .reset_index()
    )

    # Untuk grafik utama: gunakan m=1.5 (nilai m terbaik) → variasi per c
    best_m   = 1.5
    sub_m    = summary[summary["m"] == best_m].copy()

    # Untuk grafik kedua: c=2 (c terbaik) → variasi per m
    best_c   = 2
    sub_c    = summary[summary["c"] == best_c].copy()

    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(
        "Perbandingan Indeks Validitas FCM\n(kiri: variasi c pada m=1.5 · kanan: variasi m pada c=2)",
        fontsize=13, weight="bold", y=0.98,
    )

    # Plot 1: FPC vs c
    axes[0, 0].plot(sub_m["c"], sub_m["partition_coefficient"],
                    marker="o", color="#2563EB", lw=2.2, ms=7)
    axes[0, 0].axvline(x=2, color="#DC2626", lw=1.4, ls="--", label="c=2 terpilih")
    axes[0, 0].set_title("Fuzzy Partition Coefficient (FPC)\n↑ semakin tinggi = semakin baik", fontsize=10)
    axes[0, 0].set_xlabel("Jumlah klaster (c)")
    axes[0, 0].set_ylabel("FPC")
    axes[0, 0].set_xticks(sub_m["c"])
    axes[0, 0].legend(fontsize=8.5)
    axes[0, 0].grid(alpha=0.25)

    # Plot 2: Xie-Beni vs c (clip agar c=4,5 yang meledak tidak merusak skala)
    xb_plot = sub_m.copy()
    xb_plot["xie_beni_clipped"] = xb_plot["xie_beni"].clip(upper=2.0)
    axes[0, 1].plot(xb_plot["c"], xb_plot["xie_beni_clipped"],
                    marker="o", color="#DC2626", lw=2.2, ms=7)
    axes[0, 1].axvline(x=2, color="#2563EB", lw=1.4, ls="--", label="c=2 terpilih")
    # Tanda nilai outlier (c≥4 yang dipotong)
    for _, r in xb_plot[xb_plot["xie_beni"] > 2.0].iterrows():
        axes[0, 1].annotate(
            f"XBI≫\n({r['xie_beni']:.2g})",
            xy=(r["c"], 2.0), xytext=(r["c"] + 0.05, 1.8),
            fontsize=7.5, color="#DC2626",
        )
    axes[0, 1].set_title("Xie-Beni Index (XBI)\n↓ semakin rendah = semakin baik", fontsize=10)
    axes[0, 1].set_xlabel("Jumlah klaster (c)")
    axes[0, 1].set_ylabel("Xie-Beni (dipotong di 2.0 untuk keterbacaan)")
    axes[0, 1].set_xticks(sub_m["c"])
    axes[0, 1].legend(fontsize=8.5)
    axes[0, 1].grid(alpha=0.25)

    # Plot 3: FPC vs m (c=2)
    axes[1, 0].plot(sub_c["m"], sub_c["partition_coefficient"],
                    marker="s", color="#2563EB", lw=2.2, ms=7)
    axes[1, 0].axvline(x=1.5, color="#DC2626", lw=1.4, ls="--", label="m=1.5 terpilih")
    axes[1, 0].set_title("FPC vs Fuzzifier (m) pada c=2\n↑ semakin tinggi = semakin baik", fontsize=10)
    axes[1, 0].set_xlabel("Nilai fuzzifier (m)")
    axes[1, 0].set_ylabel("FPC")
    axes[1, 0].set_xticks(sub_c["m"])
    axes[1, 0].legend(fontsize=8.5)
    axes[1, 0].grid(alpha=0.25)

    # Plot 4: MPC vs m (c=2)
    axes[1, 1].plot(sub_c["m"], sub_c["modified_partition_coefficient"],
                    marker="s", color="#16A34A", lw=2.2, ms=7)
    axes[1, 1].axvline(x=1.5, color="#DC2626", lw=1.4, ls="--", label="m=1.5 terpilih")
    axes[1, 1].set_title("Modified Partition Coefficient (MPC) vs m\n↑ semakin tinggi = semakin baik", fontsize=10)
    axes[1, 1].set_xlabel("Nilai fuzzifier (m)")
    axes[1, 1].set_ylabel("MPC")
    axes[1, 1].set_xticks(sub_c["m"])
    axes[1, 1].legend(fontsize=8.5)
    axes[1, 1].grid(alpha=0.25)

    fig.tight_layout(pad=2.0, rect=[0, 0, 1, 0.96])

    out = FIG_DIR / "grafik_indeks_validitas.png"
    fig.savefig(out, dpi=220, bbox_inches="tight", facecolor="white")
    print(f"Grafik indeks validitas disimpan → {out}")
    plt.close(fig)

In [9]:
def main():
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    plot_centroid_heatmap()
    plot_validity_indices()

### Jalankan Pipeline

In [10]:
if __name__ == "__main__":
    main()

Heatmap centroid disimpan → C:\muti\SMT 6\PKK_PROJECT AKHIR\pkk-stunting-fcm-clustering\outputs\figures\heatmap_centroid.png
Grafik indeks validitas disimpan → C:\muti\SMT 6\PKK_PROJECT AKHIR\pkk-stunting-fcm-clustering\outputs\figures\grafik_indeks_validitas.png
